In [2]:
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, OrdinalEncoder, TargetEncoder, KBinsDiscretizer, PolynomialFeatures
from sklearn.feature_extraction import FeatureHasher
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV, train_test_split
from sklearn.base import BaseEstimator, TransformerMixin

import pandas as pd
import numpy as np

In [3]:
df = fetch_openml("titanic", version=1, as_frame=True).frame

print(df.columns)
df["survived"] = df["survived"].astype(int)


display("First 5 records:", df.head())
df.shape

Index(['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket',
       'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest'],
      dtype='str')


'First 5 records:'

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


(1309, 14)

In [4]:
class FeaturesTitanic(BaseEstimator, TransformerMixin):

    def __init__(self, usar=True):
        self.usar = usar
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["FamilySize"] = X["sibsp"] + X["parch"] + 1
        X["IsAlone"] = (X["FamilySize"] == 1).astype(int)
        X["FarePerPerson"] = X["fare"] / X["FamilySize"]
        X["Title"] = X["name"].str.extract(r" ([A-Za-z]+)\.", expand=False).str.strip().replace(["Mlle", "Ms"], "Miss").replace("Mme", "Mrs")
        X["Title"] = X["Title"].where(X["Title"].isin(["Mr", "Miss", "Mrs", "Master"]), "Rare")
        X["fare"] = np.log1p(X["fare"]) # Fare tem assimetria forte, então aplicamos log para reduzir a assimetria
        # Criar mais 3 novas features a partir das atuais:
        X["IsChild"] = X['age'] < 12 # Criancas tinham prioridade?
        X["NameLength"] = X["name"].str.len() # Sobrenomes maiores = nobreza?
        X["Embarked_Alone"] = X["embarked"].astype(str) + "_" + X["IsAlone"].astype(str) # Viajantes que embarcaram sozinhos
        return X.drop(columns=["name"])

def make_model(num_cols=None, cat_cols=None):
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler())
    ])

    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', min_frequency=10))
    ])

    preprocessor = ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ])
    
    return Pipeline([
        ('feat', FeaturesTitanic()),
        ('preprocessor', preprocessor),
        ('clf', LogisticRegression(max_iter=1000))
    ])

## Split

In [ ]:
X = df.drop(columns=['survived', 'home.dest', 'ticket', 'body', 'boat', 'cabin'])
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cv = StratifiedKFold(5, shuffle=True, random_state=42) 

print(X_train[:4])
print(y_train[:4])

<class 'pandas.DataFrame'>
<class 'pandas.Series'>
      pclass                                           name     sex   age  \
999        3              McCarthy, Miss. Catherine 'Katie'  female   NaN   
392        2  del Carlo, Mrs. Sebastiano (Argenia Genovesi)  female  24.0   
628        3              Andersson, Miss. Sigrid Elisabeth  female  11.0   
1165       3                               Saad, Mr. Khalil    male  25.0   

      sibsp  parch     fare embarked  
999       0      0   7.7500        Q  
392       1      0  27.7208        C  
628       4      2  31.2750        S  
1165      0      0   7.2250        C  
999     1
392     1
628     0
1165    0
Name: survived, dtype: int64


## Escalonamento

In [ ]:
X_train_fe = FeaturesTitanic().fit_transform(X_train)

num_cols = X_train_fe.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X_train_fe.select_dtypes(include=['category', 'object', 'str']).columns

print("Numerical columns:", list(num_cols))
print("Categorical columns:", list(cat_cols))

Numerical columns: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'FamilySize', 'IsAlone', 'FarePerPerson', 'NameLength']
Categorical columns: ['sex', 'embarked', 'Title', 'Embarked_Alone']


In [ ]:
scores = cross_val_score(make_model(num_cols, cat_cols), X_train, y_train, cv=cv, scoring='accuracy') # cv é o valor de k-fold cross validation
print(f"{scores.mean().round(4)} (+/- {scores.std().round(4) * 2})")
pipe = make_model(num_cols, cat_cols).fit(X_train, y_train)

0.8014 (+/- 0.0324)


In [ ]:
names = pipe.named_steps["preprocessor"].get_feature_names_out()
coefs = pipe.named_steps['clf'].coef_[0]
pd.Series(coefs, index=names).sort_values()

cat__Title_Mr             -0.945633
num__pclass               -0.702049
cat__sex_male             -0.679200
cat__Title_Rare           -0.676042
num__age                  -0.385700
num__FamilySize           -0.322281
num__sibsp                -0.319333
num__IsAlone              -0.268550
cat__Embarked_Alone_Q_1   -0.185921
cat__embarked_S           -0.175946
cat__embarked_Q           -0.146587
cat__Embarked_Alone_S_1   -0.114982
cat__Embarked_Alone_S_0   -0.060964
num__FarePerPerson        -0.043696
cat__Embarked_Alone_C_1   -0.036309
num__parch                -0.002949
cat__Embarked_Alone_Q_0    0.039334
cat__Title_Miss            0.221621
cat__embarked_C            0.253871
cat__Title_Mrs             0.285051
cat__Embarked_Alone_C_0    0.290180
num__NameLength            0.356805
num__fare                  0.399994
cat__sex_female            0.610538
cat__Title_Master          1.046341
dtype: float64

In [ ]:
grid = {
    "preprocessor__num__imputer__strategy": ["median", "mean"],
    "preprocessor__cat__imputer__strategy": ["most_frequent", "constant"],
    "preprocessor__cat__encoder": [
        OneHotEncoder(handle_unknown='ignore', sparse_output=False),
        OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
        TargetEncoder(),
        FeatureHasher(n_features=10, input_type='string')
    ],
    "clf__C": [0.1, 1.0, 10.0],
}

gs = GridSearchCV(pipe, grid, cv=cv, scoring="accuracy", n_jobs=-1)
gs.fit(X_train, y_train)

print("Melhor encoder/parâmetros:", gs.best_params_)
print("Melhor acurácia:", round(gs.best_score_, 4))

Melhor encoder/parâmetros: {'clf__C': 10.0, 'preprocessor__cat__encoder': TargetEncoder(), 'preprocessor__cat__imputer__strategy': 'most_frequent', 'preprocessor__num__imputer__strategy': 'mean'}
Melhor acurácia: 0.809


### Discretização e Interação

In [ ]:
# Discretização é o processo de transformar variáveis contínuas em variáveis categóricas, dividindo o intervalo de valores em bins (faixas).
# Isso pode ser útil para capturar relações não lineares entre a variável contínua e a variável alvo, além de reduzir o impacto de outliers.
KBinsDiscretizer(
    n_bins=5,
    encode='onehot',
    strategy='quantile'
)

# A transformação polinomial é uma técnica de engenharia de recursos que cria novas características a partir das existentes,
# elevando-as a potências e combinando-as. Isso permite que modelos lineares capturem relações não lineares entre as variáveis independentes e a variável dependente.
PolynomialFeatures(
    degree=2,
    interaction_only=True,
    include_bias=False
)

,"interaction_only interaction_only: bool, default=FalseIf `True`, only interaction features are produced: features that areproducts of at most `degree` *distinct* input features, i.e. terms withpower of 2 or higher of the same input feature are excluded:- included: `x[0]`, `x[1]`, `x[0] * x[1]`, etc.- excluded: `x[0] ** 2`, `x[0] ** 2 * x[1]`, etc.",True
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a linear model).",False
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",2
,"order order: {'C', 'F'}, default='C'Order of output array in the dense case. `'F'` order is faster tocompute, but may slow down subsequent estimators... versionadded:: 0.21",'C'


In [ ]:
LIMIAR = 0.002                                        
candidatas = [("FamilySize", "num"), ("IsAlone", "num"),
              ("FarePerPerson", "num"), ("Title", "cat")]

ref = scores.mean()

num_base, cat_base = ["pclass", "age", "sibsp", "parch", "fare"], ["sex", "embarked"]
num, cat = list(num_base), list(cat_base)
for nome, tipo in candidatas:                     
    n2, c2 = (num + [nome], cat) if tipo == "num" else (num, cat + [nome])
    s = cross_val_score(make_model(n2, c2), X_train, y_train, cv=cv)
    ganho = s.mean() - ref
    print(f"{nome:>14}  {s.mean():.3f} +/- {s.std():.3f}  ganho {ganho:+.3f}")
    if ganho > LIMIAR:
        num, cat, ref = n2, c2, s.mean()                

print("\nfeatures mantidas:", num, cat)

# Teste: usado UMA vez, no final, com o conjunto escolhido
final = make_model(num, cat).fit(X_train, y_train)
print(f"acuracia no teste: {final.score(X_test, y_test):.3f}")

    FamilySize  0.783 +/- 0.013  ganho -0.018
       IsAlone  0.780 +/- 0.015  ganho -0.021
 FarePerPerson  0.781 +/- 0.013  ganho -0.020
         Title  0.795 +/- 0.015  ganho -0.007

features mantidas: ['pclass', 'age', 'sibsp', 'parch', 'fare'] ['sex', 'embarked']
acuracia no teste: 0.813


In [ ]:
y_pred_test = pipe.predict(X_test)
y_prob_test = pipe.predict_proba(X_test)

df_resultados = pd.DataFrame({
    'Real': y_test.values,
    'Previsto': y_pred_test,
    'Prob_Sobreviver': y_prob_test[:, 1].round(3),
    'Acertou?': y_test.values == y_pred_test
})

print("Primeiros 10 passageiros do teste:")
display(df_resultados.head(10))

print(f"\nAcurácia no Teste: {(y_test.values == y_pred_test).mean():.1%}")

novo_passageiro = pd.DataFrame([{
    "pclass": 3,
    "sex": "male",
    "age": 25,
    "sibsp": 1,
    "parch": 0,
    "fare": 7.5,
    "embarked": "S",
    "name": "Silva, Mr. Joao",
}])

classe_novo = pipe.predict(novo_passageiro)[0]
prob_novo = pipe.predict_proba(novo_passageiro)[0, 1]

print(f"Previsão: {'Sobreviveu (1)' if classe_novo == 1 else 'Não sobreviveu (0)'}")
print(f"Probabilidade de sobrevivência: {prob_novo:.1%}")

Primeiros 10 passageiros do teste:


,Real,Previsto,Prob_Sobreviver,Acertou?
0,1,1,0.544,True
1,1,1,0.551,True
2,0,0,0.104,True
3,0,0,0.098,True
4,0,0,0.059,True
5,0,0,0.101,True
6,0,0,0.276,True
7,1,0,0.265,False
8,1,1,0.971,True
9,0,0,0.103,True



Acurácia no Teste: 82.4%
Previsão: Não sobreviveu (0)
Probabilidade de sobrevivência: 5.8%
